# TMI isocenter: run the project in Google Colab

This notebook trains a convolutional neural network that predicts the **field geometry** (isocenter positions and jaw apertures) of Total Marrow Irradiation radiotherapy plans, and shows its predictions on the test patients.

### Before you start
1. **Use a GPU runtime:** menu *Runtime → Change runtime type → T4 GPU → Save*. Training also runs on CPU, but it is ~5× slower.
2. **Run all cells:** menu *Runtime → Run all*.
3. The first cell **restarts the session** (Colab shows a *"session crashed"* or *"restarting"* message: this is expected). When it happens, do **Runtime → Run all** again. The first cell then skips the installation and the rest of the notebook runs.

⏱️ Expected time: ~5 min for setup, then ~2 min per training epoch on a T4 GPU (~11 min on CPU).

⚠️ Colab deletes all files when the session ends. Use **Step 7** to save your results to Google Drive.

## Step 1: Install conda

The project requires **Python 3.11**, while Colab ships a newer Python. We install `conda` to create a separate Python 3.11 environment. **This cell restarts the session the first time it runs.**

In [ ]:
!pip install -q "condacolab<0.2"
import condacolab

condacolab.install()  # restarts the session the first time; does nothing afterwards

## Settings

Change these values to experiment, then run the notebook again from here (*Runtime → Run cell and below*).

In [ ]:
MODEL = "body"  # "body": patients without isocenters on the arms | "arms": patients with isocenters on the arms
MAX_EPOCHS = 10  # maximum training epochs (early stopping may stop earlier). The full training uses 50
N_PLOTS = 5  # number of test-patient predictions to display at the end

## Step 2: Download the code and the data

In [ ]:
import os

REPO_DIR = "/content/tmi-isocenter"

if not os.path.exists(REPO_DIR):
    !git clone --branch daihs --depth 1 https://github.com/nlambriICH/tmi-isocenter.git {REPO_DIR}
else:
    print("Repository already downloaded.")

%cd {REPO_DIR}

In [ ]:
# Download the dataset shared by the instructor on Google Drive
try:
    import gdown
except ImportError:
    !pip install -q gdown
    import gdown

DATA_FOLDER_ID = "1QRU3aQf_wFEVIWrHMJkHisoK8FxRQQUo"
DATA_DIR = f"{REPO_DIR}/data/5_355"

if not os.path.exists(f"{DATA_DIR}/raw/ptv_imgs2D.npz"):
    gdown.download_folder(id=DATA_FOLDER_ID, output=DATA_DIR, quiet=True, use_cookies=False)

expected_files = ["patient_info.csv"] + [
    f"raw/{name}"
    for name in [
        "ptv_imgs2D.npz", "ptv_masks2D.npz", "brain_masks2D.npz", "lungs_masks2D.npz",
        "liver_masks2D.npz", "intestine_masks2D.npz", "bladder_masks2D.npz",
        "isocenters_pix.npy", "jaws_X_pix.npy", "jaws_Y_pix.npy", "angles.npy",
    ]
]
missing = [f for f in expected_files if not os.path.exists(f"{DATA_DIR}/{f}")]
assert not missing, f"Missing data files: {missing}. Re-run this cell or ask the instructor."
print("Dataset ready ✅")

In [ ]:
# Create the configuration file from the template, using the settings above
import json

config_path = f"{REPO_DIR}/src/config/constants.json"
with open(f"{REPO_DIR}/src/config/constants.example.json") as f:
    config = json.load(f)

config["model"] = MODEL
config["coll_5_355"] = True  # the shared dataset uses the 5°/355° pelvis collimator configuration
config["num_workers"] = 2  # Colab provides 2 CPU cores

with open(config_path, "w") as f:
    json.dump(config, f, indent=4)
print(f"Configuration saved: model = {MODEL!r}")

## Step 3: Create the Python 3.11 environment and install the dependencies

All project commands run inside this environment with `conda run -n tmi_env ...`. Takes ~3-5 min, only the first time.

In [ ]:
ENV_READY_FLAG = "/content/.tmi_env_ready"

if not os.path.exists(ENV_READY_FLAG):
    !conda create -y -q -n tmi_env python=3.11
    !conda run -n tmi_env pip install -q -r requirements.txt
    !conda run -n tmi_env pip install -q -e .
    # PyTorch Lightning 2.0 needs pkg_resources, removed from recent setuptools
    !conda run -n tmi_env pip install -q "setuptools<70"
    open(ENV_READY_FLAG, "w").close()
else:
    print("Environment already installed.")

!conda run -n tmi_env python -c "import torch; print('Python OK | PyTorch', torch.__version__, '| GPU available:', torch.cuda.is_available())"

## Step 4: Pre-process the data

Combines the masks into 3-channel 512×512 images and transforms the labels accordingly (output in `data/5_355/interim`).

In [ ]:
if not os.path.exists(f"{DATA_DIR}/interim/masks2D.npy"):
    !conda run -n tmi_env python -m src.data.processing
print("Pre-processing done ✅")

## Step 5: Train and test the model

Optional: run the next cell first to open **TensorBoard** and follow the training losses (click the refresh icon in TensorBoard to update the charts). If it does not work, skip it.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir lightning_logs

In [ ]:
# Trains the model, then evaluates it on the test patients and saves the plots
!conda run -n tmi_env --no-capture-output python -u -m src.train_cnn --max-epochs {MAX_EPOCHS}

## Step 6: Look at the results

Each figure shows the predicted fields (rectangles) and isocenters on a **test patient** the model has never seen. The ground truth plans are saved next to them, in `img/test/ground_truth`.

In [ ]:
import glob

from IPython.display import Image, display

log_dir = f"{REPO_DIR}/lightning_logs/{MODEL}_model_5_355"
versions = sorted(glob.glob(f"{log_dir}/version_*"), key=lambda p: int(p.rsplit("_", 1)[1]))
assert versions, "No training run found: run Step 5 first."
run_dir = versions[-1]
print("Latest run:", run_dir)
print("Checkpoints:", glob.glob(f"{run_dir}/checkpoints/*.ckpt"))

for img_path in sorted(glob.glob(f"{run_dir}/img/test/predicted/*.png"))[:N_PLOTS]:
    print(os.path.basename(img_path))
    display(Image(img_path))

## Step 7: Save the results to Google Drive (optional)

Copies the latest run (checkpoint, TensorBoard logs and plots) to `MyDrive/tmi-isocenter-results`. Colab asks you to authorize access to your Drive.

In [ ]:
import shutil

from google.colab import drive

drive.mount("/content/drive")
destination = f"/content/drive/MyDrive/tmi-isocenter-results/{MODEL}_{os.path.basename(run_dir)}"
shutil.copytree(run_dir, destination, dirs_exist_ok=True)
print("Results saved to", destination)

## What to try next

- Change `MODEL` or `MAX_EPOCHS` in **Settings** and run again from there: each run is saved in a new `version_N` folder.
- Edit the code directly in Colab (folder icon on the left → `tmi-isocenter/src`), e.g. the network in `src/models/cnn.py`, the learning rate in `src/modules/`, the augmentation in `src/data/augmentation.py`, then re-run Step 5.